# PostgreSQL Avançado 05: Performance em Produção

Este notebook aborda os principais gargalos de performance em PostgreSQL de produção -- connection pooling, VACUUM/MVCC, bulk operations, advisory locks e monitoramento -- e fecha com um caso real de sistema que trava no horário de pico, diagnosticado e resolvido passo a passo.

**Pré-requisitos:**
- Notebook 01: Índices e EXPLAIN
- Notebook 02: Queries Avançadas
- Notebook 04: Concorrência e Transações

**Schema utilizado:** Sistema de crédito com propostas, clientes, consultas externas, decisões e parcelas.

---

## Contexto: O Problema de Produção

Um sistema de crédito funciona bem em desenvolvimento, mas em produção apresenta lentidão extrema às 10h quando todos os analistas abrem a fila de propostas. O banco de dados tem milhões de registros e recebe picos de 200 conexões simultâneas.

Este notebook cobre as ferramentas para diagnosticar e resolver esse tipo de problema.

## Arquitetura: Fluxo de uma Requisição em Produção

![Fluxo de uma requisição em produção](assets/05-performance-producao.png)

Cada requisição passa por 4 pontos potenciais de gargalo:

1. **Pool esgotado** -- `pool_size` muito pequeno ou conexões vazando (não devolvidas ao pool)
2. **Cache miss** -- `shared_buffers` insuficiente ou Seq Scan carregando páginas demais
3. **I/O lento** -- disco saturado ou table bloat por falta de VACUUM
4. **Conexão não devolvida** -- leak de conexão bloqueia outros requests

In [1]:
# Dependências
# !pip install psycopg2-binary sqlalchemy faker

import time
import io
import csv
import random
from datetime import date, timedelta
from decimal import Decimal

import psycopg2
import psycopg2.extras
from sqlalchemy import create_engine, text, pool

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "fintech_kb",
    "user": "postgres",
    "password": "postgres",
}
DSN = "host={host} port={port} dbname={dbname} user={user} password={password}".format(**DB_CONFIG)

try:
    conn = psycopg2.connect(DSN)
    cur = conn.cursor()
    cur.execute("SELECT version()")
    print(cur.fetchone()[0])
    conn.close()
    DB_DISPONIVEL = True
except Exception as e:
    print(f"PostgreSQL não disponível: {e}")
    print("Inicie com: docker run -d -e POSTGRES_USER=postgres -e POSTGRES_PASSWORD=postgres "
          "-e POSTGRES_DB=fintech_kb -p 5432:5432 postgres:16")
    DB_DISPONIVEL = False

PostgreSQL 16.13 on x86_64-pc-linux-musl, compiled by gcc (Alpine 15.2.0) 15.2.0, 64-bit


In [2]:
# Criação do schema do sistema de crédito
DDL = """
CREATE TABLE IF NOT EXISTS clientes (
    id          SERIAL PRIMARY KEY,
    cpf         CHAR(11)       NOT NULL UNIQUE,
    nome        VARCHAR(200)   NOT NULL,
    renda_mensal NUMERIC(12,2) NOT NULL,
    created_at  TIMESTAMPTZ    NOT NULL DEFAULT now()
);

CREATE TABLE IF NOT EXISTS propostas (
    id         SERIAL PRIMARY KEY,
    numero     VARCHAR(20)    NOT NULL UNIQUE,
    cliente_id INT            NOT NULL REFERENCES clientes(id),
    valor      NUMERIC(12,2)  NOT NULL,
    status     VARCHAR(20)    NOT NULL DEFAULT 'pendente',
    score      SMALLINT,
    created_at TIMESTAMPTZ    NOT NULL DEFAULT now()
);

CREATE TABLE IF NOT EXISTS consultas_externas (
    id           SERIAL PRIMARY KEY,
    proposta_id  INT          NOT NULL REFERENCES propostas(id),
    provider     VARCHAR(50)  NOT NULL,
    response_data JSONB,
    status       VARCHAR(20)  NOT NULL,
    created_at   TIMESTAMPTZ  NOT NULL DEFAULT now()
);

CREATE TABLE IF NOT EXISTS decisoes_log (
    id               SERIAL PRIMARY KEY,
    proposta_id      INT         NOT NULL REFERENCES propostas(id),
    etapa            VARCHAR(50) NOT NULL,
    resultado        VARCHAR(20) NOT NULL,
    regras_aplicadas JSONB,
    executed_at      TIMESTAMPTZ NOT NULL DEFAULT now()
);

CREATE TABLE IF NOT EXISTS parcelas (
    id               SERIAL PRIMARY KEY,
    proposta_id      INT          NOT NULL REFERENCES propostas(id),
    numero           SMALLINT     NOT NULL,
    valor            NUMERIC(10,2) NOT NULL,
    vencimento       DATE         NOT NULL,
    status_pagamento VARCHAR(20)  NOT NULL DEFAULT 'pendente',
    pago_em          TIMESTAMPTZ
);
"""

if DB_DISPONIVEL:
    conn = psycopg2.connect(DSN)
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(DDL)
    conn.close()
    print("Schema criado com sucesso")

Schema criado com sucesso


## 1. Connection Pooling

O PostgreSQL cria um **processo OS por conexão**, cada um ocupando ~10 MB de RAM. Em uma aplicação web com 500 requests/s, abrir uma conexão por request é inviável -- o overhead de criação (parse TCP, autenticação, fork) se acumula rapidamente.

### PgBouncer

PgBouncer é um pooler externo que fica entre a aplicação e o PostgreSQL, reutilizando conexões reais:

```ini
# pgbouncer.ini
[databases]
fintech_kb = host=localhost port=5432 dbname=fintech_kb

[pgbouncer]
listen_port = 6432
listen_addr = 0.0.0.0
auth_type = md5
auth_file = /etc/pgbouncer/userlist.txt

pool_mode = transaction   # Conexão devolvida ao pool após cada COMMIT/ROLLBACK
max_client_conn = 1000    # Conexões lógicas (app -> PgBouncer)
default_pool_size = 25    # Conexões reais (PgBouncer -> PostgreSQL)
min_pool_size = 5
reserve_pool_size = 5
```

Os outros modos (`session` e `statement`) existem para compatibilidade e para workloads atípicos, mas `transaction` é o padrão para APIs REST -- cada request abre, executa e faz commit, liberando a conexão imediatamente.

### Dimensionamento do pool

A fórmula clássica do PostgreSQL wiki:

```
pool_size = (número_de_cores * 2) + número_de_spindles
```

Para 4 cores + SSD (1 spindle efetivo): `(4 * 2) + 1 = 9`, arredondado para 10. Mais conexões além disso competem por CPU e causam context switching, reduzindo throughput.

In [3]:
# Configuração do pool no SQLAlchemy
# Em produção o DSN aponta para o PgBouncer (porta 6432), não diretamente para o PostgreSQL

SQLALCHEMY_URL = "postgresql://{user}:{password}@{host}:{port}/{dbname}".format(**DB_CONFIG)

engine = create_engine(
    SQLALCHEMY_URL,
    # pool_size: conexões mantidas abertas permanentemente
    # Fórmula: (core_count * 2) + effective_spindle_count
    pool_size=10,

    # max_overflow: conexões extras criadas em picos (além do pool_size)
    # São fechadas quando o pico passa
    max_overflow=5,

    # pool_timeout: tempo máximo de espera por uma conexão livre (segundos)
    # Lançar TimeoutError é melhor do que travar indefinidamente
    pool_timeout=30,

    # pool_recycle: forçar renovação da conexão após N segundos
    # Evita conexões 'zumbis' (keepalive do firewall pode fechar conexões ociosas)
    pool_recycle=1800,

    # pool_pre_ping: executar SELECT 1 antes de entregar conexão do pool
    # Custo: ~0.1ms por request, benefício: zero erros de conexão morta
    pool_pre_ping=True,
)

print("Engine configurado:")
print(f"  pool_size    : {engine.pool.size()}")
print(f"  max_overflow : {engine.pool._max_overflow}")
print(f"  pool_timeout : {engine.pool._timeout}s")
print(f"  pool_recycle : {engine.pool._recycle}s")

# Demonstrar reutilização de conexões
if DB_DISPONIVEL:
    tempos = []
    for i in range(5):
        t0 = time.time()
        with engine.connect() as conn:
            conn.execute(text("SELECT 1"))
        tempos.append((time.time() - t0) * 1000)

    print(f"\nTempo de aquisição de conexão (pool aquecido):")
    for i, t in enumerate(tempos):
        label = "(cold start)" if i == 0 else "(pool hit)  "
        print(f"  Chamada {i+1} {label}: {t:.2f}ms")

Engine configurado:
  pool_size    : 10
  max_overflow : 5
  pool_timeout : 30s
  pool_recycle : 1800s

Tempo de aquisição de conexão (pool aquecido):
  Chamada 1 (cold start): 13.09ms
  Chamada 2 (pool hit)  : 1.19ms
  Chamada 3 (pool hit)  : 0.70ms
  Chamada 4 (pool hit)  : 0.62ms
  Chamada 5 (pool hit)  : 0.70ms


## 2. VACUUM e AUTOVACUUM

### MVCC e dead tuples

O PostgreSQL usa **MVCC (Multi-Version Concurrency Control)**: ao atualizar uma linha, o banco cria uma nova versão e marca a antiga como obsoleta. Transações concorrentes continuam vendo a versão antiga até finalizarem -- sem locks de leitura.

O custo: versões antigas (dead tuples) acumulam no disco. A tabela cresce (table bloat), queries varrem mais páginas, e o planner trabalha com estatísticas desatualizadas.

### AUTOVACUUM

O daemon autovacuum roda em background e reclaima espaço de dead tuples, atualiza estatísticas e previne transaction ID wraparound. Na maioria dos casos funciona sem intervenção, mas tabelas de alta escrita podem gerar dead tuples mais rápido do que o autovacuum consegue processar.

Para verificar:

```sql
SELECT relname, n_live_tup, n_dead_tup,
       ROUND(100.0 * n_dead_tup / NULLIF(n_live_tup + n_dead_tup, 0), 1) AS dead_pct,
       last_autovacuum, autovacuum_count
FROM pg_stat_user_tables
WHERE n_dead_tup > 1000
ORDER BY n_dead_tup DESC;
```

**Sinal de alerta:** `dead_pct > 20%` ou `last_autovacuum` com timestamp antigo em tabelas de alta escrita.

### VACUUM vs VACUUM FULL

`VACUUM tabela` marca espaço como reutilizável **sem lock** -- pode rodar em produção a qualquer momento, é o que o autovacuum faz. `VACUUM ANALYZE` faz o mesmo e ainda atualiza estatísticas, recomendado após carga massiva.

`VACUUM FULL tabela` reescreve a tabela inteira no disco, compactando-a fisicamente. Exige `ACCESS EXCLUSIVE LOCK` (bloqueia todas as operações), então só deve ser usado em janelas de manutenção quando o bloat já saiu do controle e a tabela ocupa muito mais espaço do que deveria.

In [4]:
# Monitorar dead tuples e autovacuum
SQL_VACUUM_STATUS = """
SELECT
    relname                                                           AS tabela,
    n_live_tup                                                        AS vivos,
    n_dead_tup                                                        AS mortos,
    ROUND(
        100.0 * n_dead_tup / NULLIF(n_live_tup + n_dead_tup, 0), 1
    )                                                                 AS dead_pct,
    autovacuum_count,
    to_char(last_autovacuum, 'YYYY-MM-DD HH24:MI')                    AS ultimo_autovacuum,
    to_char(last_autoanalyze, 'YYYY-MM-DD HH24:MI')                   AS ultimo_autoanalyze
FROM pg_stat_user_tables
ORDER BY n_dead_tup DESC NULLS LAST;
"""

if DB_DISPONIVEL:
    with engine.connect() as conn:
        rows = conn.execute(text(SQL_VACUUM_STATUS)).fetchall()

    print(f"{'Tabela':<25} {'Vivos':>8} {'Mortos':>8} {'Dead%':>7} {'AVac':>5} {'Último VACUUM':<20}")
    print("-" * 80)
    for row in rows:
        dead_pct = row[3] if row[3] is not None else 0.0
        alerta = " ALERTA" if float(dead_pct) > 20 else ""
        print(
            f"{row[0]:<25} {row[1]:>8} {row[2]:>8} {float(dead_pct):>6.1f}%"
            f" {row[4] or 0:>5} {str(row[5] or 'nunca'):<20}{alerta}"
        )

    if not rows:
        print("Nenhuma tabela de usuário encontrada ainda.")

Tabela                       Vivos   Mortos   Dead%  AVac Último VACUUM       
--------------------------------------------------------------------------------
bureau_consultas                 6        1   14.3%     0 nunca               
documentos                       2        1   33.3%     0 nunca                ALERTA
propostas_particoes_2025_07        0        0    0.0%     0 nunca               
propostas_particoes_2026_05        0        0    0.0%     0 nunca               
bureau_consultas_2025_02         1        0    0.0%     0 nunca               
propostas_particoes_2026_07        0        0    0.0%     0 nunca               
clientes_historico               0        0    0.0%     0 nunca               
propostas_particoes_2026_01        1        0    0.0%     0 nunca               
propostas_particoes_2026_04        0        0    0.0%     0 nunca               
parcelas                         0        0    0.0%     0 nunca               
propostas_particoes_2025_03      

In [5]:
# Simulação: UPDATE em massa gera dead tuples, depois VACUUM recupera o espaço
if DB_DISPONIVEL:
    conn_raw = psycopg2.connect(DSN)
    conn_raw.autocommit = True
    cur = conn_raw.cursor()

    # Limpar tabelas na ordem correta (FKs)
    cur.execute("DELETE FROM parcelas")
    cur.execute("DELETE FROM decisoes_log")
    cur.execute("DELETE FROM consultas_externas")
    cur.execute("DELETE FROM propostas")
    cur.execute("DELETE FROM clientes")

    # Inserir clientes de teste
    cur.execute("""
        INSERT INTO clientes (cpf, nome, renda_mensal)
        SELECT
            LPAD(generate_series::text, 11, '0'),
            'Cliente ' || generate_series,
            (random() * 10000 + 1000)::numeric(12,2)
        FROM generate_series(1, 10000);
    """)
    print("10.000 clientes inseridos")

    # Verificar dead tuples antes
    cur.execute("SELECT n_live_tup, n_dead_tup FROM pg_stat_user_tables WHERE relname = 'clientes'")
    antes = cur.fetchone()
    print(f"Antes do UPDATE: {antes[0]} vivos, {antes[1]} mortos")

    # UPDATE em massa: cada linha atualizada gera uma dead tuple
    cur.execute("UPDATE clientes SET renda_mensal = renda_mensal * 1.10")

    # ANALYZE atualiza estatísticas sem resetar contadores
    cur.execute("ANALYZE clientes")
    cur.execute("SELECT n_live_tup, n_dead_tup FROM pg_stat_user_tables WHERE relname = 'clientes'")
    depois = cur.fetchone()
    print(f"Após UPDATE:     {depois[0]} vivos, {depois[1]} mortos")

    # VACUUM recupera o espaço das dead tuples
    cur.execute("VACUUM ANALYZE clientes")
    cur.execute("SELECT n_live_tup, n_dead_tup FROM pg_stat_user_tables WHERE relname = 'clientes'")
    apos_vacuum = cur.fetchone()
    print(f"Após VACUUM:     {apos_vacuum[0]} vivos, {apos_vacuum[1]} mortos")

    conn_raw.close()

10.000 clientes inseridos
Antes do UPDATE: 10000 vivos, 10000 mortos
Após UPDATE:     10000 vivos, 10000 mortos
Após VACUUM:     10000 vivos, 0 mortos


## 3. Bulk Operations

Cada INSERT individual paga round-trip de rede, parse/planejamento, WAL flush e atualização de índices. Para 100.000 linhas, esse custo se multiplica. As alternativas, do mais simples ao mais rápido:

| Abordagem | Velocidade relativa | Quando usar |
|-----------|-----------|-------------|
| INSERT 1-a-1 | 1x (baseline) | Nunca em bulk |
| `executemany` | 10-50x | Batch pequeno com lógica Python entre inserções |
| `execute_values` | 20-80x | Batch grande; preferido no psycopg2 |
| `COPY` (`copy_from` / `copy_expert`) | 50-200x | Carga massiva; bypassa quase todo o overhead |

### UPSERT (INSERT ... ON CONFLICT)

Trata conflitos diretamente no banco, sem round-trip extra para verificar existência:

```sql
-- Atualiza se já existe
INSERT INTO propostas (numero, cliente_id, valor, status)
VALUES (%s, %s, %s, %s)
ON CONFLICT (numero)
DO UPDATE SET valor = EXCLUDED.valor, status = EXCLUDED.status;

-- Ignora silenciosamente duplicatas
INSERT INTO propostas (numero, cliente_id, valor)
VALUES (%s, %s, %s)
ON CONFLICT (numero) DO NOTHING;
```

In [6]:
# Preparar dados de propostas para benchmark
import random
import string

# Buscar IDs reais dos clientes para evitar FK violations
if DB_DISPONIVEL:
    conn_tmp = psycopg2.connect(DSN)
    cur_tmp = conn_tmp.cursor()
    cur_tmp.execute("SELECT MIN(id), MAX(id) FROM clientes")
    MIN_CLIENTE_ID, MAX_CLIENTE_ID = cur_tmp.fetchone()
    cur_tmp.close()
    conn_tmp.close()
    print(f"Clientes no banco: IDs de {MIN_CLIENTE_ID} a {MAX_CLIENTE_ID}")
else:
    MIN_CLIENTE_ID, MAX_CLIENTE_ID = 1, 10000

def gerar_propostas(n, offset=0):
    """Gera n propostas como lista de tuplas (numero, cliente_id, valor, status, score)."""
    statuses = ['pendente', 'aprovada', 'recusada', 'em_analise']
    return [
        (
            f"PROP-{offset + i:08d}",
            random.randint(MIN_CLIENTE_ID, MAX_CLIENTE_ID),
            round(random.uniform(500, 50000), 2),
            random.choice(statuses),
            random.randint(300, 850),
        )
        for i in range(n)
    ]

N = 10_000  # Reduzido para demo rápida; em produção teste com 100k
dados = gerar_propostas(N)
print(f"{N} propostas geradas em memória")
print(f"Exemplo: {dados[0]}")

Clientes no banco: IDs de 25001 a 35000
10000 propostas geradas em memória
Exemplo: ('PROP-00000000', 32381, 21958.42, 'recusada', 333)


In [7]:
# Benchmark: INSERT 1-a-1 vs execute_values vs COPY

INSERT_SQL = "INSERT INTO propostas (numero, cliente_id, valor, status, score) VALUES (%s, %s, %s, %s, %s)"

resultados_bulk = {}

if DB_DISPONIVEL:
    conn_raw = psycopg2.connect(DSN)

    # --- Método 1: INSERT 1-a-1 (apenas 500 linhas para não demorar demais) ---
    SAMPLE = 500
    cur = conn_raw.cursor()
    cur.execute("DELETE FROM propostas")
    conn_raw.commit()

    t0 = time.time()
    for row in dados[:SAMPLE]:
        cur.execute(INSERT_SQL, row)
    conn_raw.commit()
    t1 = time.time()
    tempo_1a1 = t1 - t0
    taxa_1a1 = SAMPLE / tempo_1a1
    resultados_bulk["INSERT 1-a-1"] = (tempo_1a1, taxa_1a1, SAMPLE)
    print(f"INSERT 1-a-1 ({SAMPLE} linhas): {tempo_1a1:.3f}s  ({taxa_1a1:.0f} linhas/s)")

    # --- Método 2: execute_values (batch de N linhas) ---
    cur.execute("DELETE FROM propostas")
    conn_raw.commit()

    t0 = time.time()
    psycopg2.extras.execute_values(
        cur,
        "INSERT INTO propostas (numero, cliente_id, valor, status, score) VALUES %s",
        dados,
        page_size=1000,  # Quantas linhas por statement SQL gerado
    )
    conn_raw.commit()
    t1 = time.time()
    tempo_ev = t1 - t0
    taxa_ev = N / tempo_ev
    resultados_bulk["execute_values"] = (tempo_ev, taxa_ev, N)
    print(f"execute_values  ({N} linhas): {tempo_ev:.3f}s  ({taxa_ev:.0f} linhas/s)")

    # --- Método 3: COPY via copy_expert (mais rápido) ---
    cur.execute("DELETE FROM propostas")
    conn_raw.commit()

    # Gerar CSV em memória
    buffer = io.StringIO()
    writer = csv.writer(buffer)
    writer.writerows(dados)
    buffer.seek(0)

    t0 = time.time()
    cur.copy_expert(
        "COPY propostas (numero, cliente_id, valor, status, score) FROM STDIN WITH CSV",
        buffer,
    )
    conn_raw.commit()
    t1 = time.time()
    tempo_copy = t1 - t0
    taxa_copy = N / tempo_copy
    resultados_bulk["COPY"] = (tempo_copy, taxa_copy, N)
    print(f"COPY            ({N} linhas): {tempo_copy:.3f}s  ({taxa_copy:.0f} linhas/s)")

    conn_raw.close()

    # Resumo comparativo (relativo ao INSERT 1-a-1)
    print("\nComparativo (relativo ao INSERT 1-a-1):")
    for metodo, (tempo, taxa, n_linhas) in resultados_bulk.items():
        speedup = taxa / taxa_1a1
        print(f"  {metodo:<20} {taxa:>8.0f} linhas/s  ({speedup:.1f}x vs 1-a-1)")

INSERT 1-a-1 (500 linhas): 0.092s  (5424 linhas/s)
execute_values  (10000 linhas): 0.256s  (39032 linhas/s)
COPY            (10000 linhas): 0.154s  (64952 linhas/s)

Comparativo (relativo ao INSERT 1-a-1):
  INSERT 1-a-1             5424 linhas/s  (1.0x vs 1-a-1)
  execute_values          39032 linhas/s  (7.2x vs 1-a-1)
  COPY                    64952 linhas/s  (12.0x vs 1-a-1)


In [8]:
# UPSERT: inserir ou atualizar se já existe
if DB_DISPONIVEL:
    conn_raw = psycopg2.connect(DSN)
    cur = conn_raw.cursor()

    # Buscar um cliente_id existente para evitar FK violation
    cur.execute("SELECT id FROM clientes LIMIT 1")
    cliente_id_existente = cur.fetchone()[0]

    # Proposta que já existe
    cur.execute("SELECT numero, status FROM propostas LIMIT 1")
    numero_existente, status_original = cur.fetchone()
    print(f"Proposta existente: {numero_existente} | status atual: {status_original}")

    # UPSERT: se conflito no número, atualiza o status
    UPSERT_SQL = """
        INSERT INTO propostas (numero, cliente_id, valor, status, score)
        VALUES (%s, %s, %s, %s, %s)
        ON CONFLICT (numero)
        DO UPDATE SET
            status = EXCLUDED.status,
            score  = EXCLUDED.score
        RETURNING id, numero, status;
    """

    cur.execute(UPSERT_SQL, (numero_existente, cliente_id_existente, 1000.00, 'aprovada', 750))
    resultado = cur.fetchone()
    conn_raw.commit()
    print(f"Após UPSERT:       id={resultado[0]} | numero={resultado[1]} | status={resultado[2]}")

    # ON CONFLICT DO NOTHING: silenciosamente ignora duplicatas
    cur.execute("""
        INSERT INTO propostas (numero, cliente_id, valor, status)
        VALUES (%s, %s, %s, %s)
        ON CONFLICT (numero) DO NOTHING
        RETURNING id;
    """, (numero_existente, cliente_id_existente, 999.00, 'pendente'))
    retorno = cur.fetchone()
    conn_raw.commit()
    print(f"ON CONFLICT DO NOTHING retornou: {retorno} (None = linha ignorada)")

    conn_raw.close()

Proposta existente: PROP-00000000 | status atual: recusada
Após UPSERT:       id=61003 | numero=PROP-00000000 | status=aprovada
ON CONFLICT DO NOTHING retornou: None (None = linha ignorada)


## 4. Advisory Locks

Advisory locks são **locks definidos pela aplicação**, sem relação com linhas ou tabelas. O PostgreSQL gerencia o ciclo de vida e garante exclusividade entre sessões.

Caso de uso típico: garantir que apenas um worker processa a proposta X, mesmo com múltiplas instâncias da aplicação.

```sql
-- Aguarda até conseguir o lock (bloqueia)
SELECT pg_advisory_lock(42);

-- Tenta sem bloquear; retorna false se já está locado
SELECT pg_try_advisory_lock(42);

-- Libera explicitamente (retorna false se não tinha o lock)
SELECT pg_advisory_unlock(42);

-- Versão transacional (liberado no COMMIT/ROLLBACK, sem unlock manual)
SELECT pg_advisory_xact_lock(42);
```

O identificador é um `bigint`. Por convenção, use o ID do recurso (proposta_id, cliente_id etc.).

### Prevenção de deadlock

Deadlocks ocorrem quando duas transações esperam uma pela outra em ordem inversa. A solução: adquirir locks na **mesma ordem** em todas as transações. Se precisar locar as propostas 1 e 2, sempre loque 1 antes de 2, independente do contexto.

In [9]:
# Advisory lock: garantir processamento exclusivo de uma proposta
import threading

# Buscar uma proposta_id existente para usar no demo
LOCK_PROPOSTA_ID = 42  # fallback
if DB_DISPONIVEL:
    conn_tmp = psycopg2.connect(DSN)
    cur_tmp = conn_tmp.cursor()
    cur_tmp.execute("SELECT id FROM propostas LIMIT 1")
    row = cur_tmp.fetchone()
    if row:
        LOCK_PROPOSTA_ID = row[0]
    cur_tmp.close()
    conn_tmp.close()
    print(f"Usando proposta_id={LOCK_PROPOSTA_ID} para demo de advisory lock")

def processar_proposta_exclusivo(proposta_id, worker_id):
    """Simula um worker que processa uma proposta com advisory lock."""
    if not DB_DISPONIVEL:
        return

    conn = psycopg2.connect(DSN)
    cur = conn.cursor()

    # pg_try_advisory_lock: não bloqueia, retorna false se outra sessão já tem o lock
    cur.execute("SELECT pg_try_advisory_lock(%s)", (proposta_id,))
    obteve_lock = cur.fetchone()[0]

    if not obteve_lock:
        print(f"  Worker {worker_id}: proposta {proposta_id} já está sendo processada por outro worker")
        cur.close()
        conn.close()
        return

    try:
        print(f"  Worker {worker_id}: iniciando processamento da proposta {proposta_id}")
        # Simula processamento
        time.sleep(0.1)
        print(f"  Worker {worker_id}: proposta {proposta_id} processada com sucesso")
    finally:
        # Liberar o lock e verificar retorno
        cur.execute("SELECT pg_advisory_unlock(%s)", (proposta_id,))
        unlocked = cur.fetchone()[0]
        if not unlocked:
            print(f"  Worker {worker_id}: AVISO - unlock retornou false (lock não estava retido)")
        conn.commit()
        cur.close()
        conn.close()

# Simular 3 workers tentando processar a mesma proposta ao mesmo tempo
print(f"\nSimulando 3 workers tentando processar a proposta {LOCK_PROPOSTA_ID}:")
threads = [
    threading.Thread(target=processar_proposta_exclusivo, args=(LOCK_PROPOSTA_ID, i))
    for i in range(1, 4)
]
for t in threads:
    t.start()
for t in threads:
    t.join()

print("\nResultado: advisory lock garante que apenas 1 worker processa por vez")

Usando proposta_id=61004 para demo de advisory lock

Simulando 3 workers tentando processar a proposta 61004:
  Worker 3: iniciando processamento da proposta 61004
  Worker 1: proposta 61004 já está sendo processada por outro worker
  Worker 2: proposta 61004 já está sendo processada por outro worker
  Worker 3: proposta 61004 processada com sucesso

Resultado: advisory lock garante que apenas 1 worker processa por vez


In [10]:
# Detectar locks ativos e sessões bloqueadas
SQL_LOCKS = """
SELECT
    a.pid,
    a.usename,
    a.state,
    a.wait_event_type,
    a.wait_event,
    EXTRACT(EPOCH FROM (now() - a.query_start))::int AS segundos,
    LEFT(a.query, 80)                                AS query_resumida
FROM pg_stat_activity a
WHERE a.pid <> pg_backend_pid()
  AND a.state IS NOT NULL
ORDER BY segundos DESC NULLS LAST;
"""

if DB_DISPONIVEL:
    with engine.connect() as conn:
        rows = conn.execute(text(SQL_LOCKS)).fetchall()

    if rows:
        print(f"{'PID':>7} {'Usuário':<12} {'Estado':<12} {'Wait':<15} {'Seg':>4} {'Query'}")
        print("-" * 90)
        for row in rows:
            print(
                f"{row[0]:>7} {str(row[1] or ''):<12} {str(row[2] or ''):<12}"
                f" {str(row[4] or ''):<15} {str(row[5] or ''):>4} {str(row[6] or '')}"
            )
    else:
        print("Nenhuma sessão ativa além desta.")

    PID Usuário      Estado       Wait             Seg Query
------------------------------------------------------------------------------------------
   8330 postgres     idle         ClientRead      23766 
SELECT
    c.nome,
    c.renda_mensal,
    primeira.numero     AS primeira_prop


## 5. Monitoramento com Views de Sistema

Quando o sistema trava, o primeiro lugar a olhar é `pg_stat_activity` -- mostra todas as conexões, o que estão fazendo e há quanto tempo. Em seguida, `pg_stat_user_tables` revela se o problema é falta de índice (muitos seq scans) ou bloat (muitas dead tuples).

### Onde começar

**`pg_stat_activity`** -- visão em tempo real. Agrupe por `state` e `wait_event` para ver se há sessões bloqueadas ou `idle in transaction` há muito tempo.

**`pg_stat_user_tables`** -- saúde das tabelas. O ratio `idx_scan / (seq_scan + idx_scan)` mostra se os índices estão sendo usados. `n_dead_tup` alto indica VACUUM atrasado.

**`pg_stat_user_indexes`** -- índices com `idx_scan = 0` após semanas de uso normal são candidatos a remoção (exceto PKs e índices de unicidade).

**`pg_stat_statements`** (extensão) -- ranking das queries mais custosas por tempo total. Base para otimização proativa. Já coberto em detalhes no notebook 01.

### Slow query log

Configurar no `postgresql.conf` ou via SQL sem reiniciar:

```sql
ALTER SYSTEM SET log_min_duration_statement = 1000;  -- milissegundos
SELECT pg_reload_conf();
```

Para logar também o plano de execução (requer a extensão `auto_explain`):

```ini
auto_explain.log_min_duration = 1000
auto_explain.log_analyze = on
```

In [11]:
# Monitorar ratio seq_scan vs idx_scan (sinal de índices faltando)
SQL_SCAN_RATIO = """
SELECT
    relname                                                              AS tabela,
    seq_scan,
    idx_scan,
    CASE
        WHEN seq_scan + idx_scan = 0 THEN NULL
        ELSE ROUND(100.0 * idx_scan / (seq_scan + idx_scan), 1)
    END                                                                  AS idx_pct,
    n_live_tup                                                           AS linhas_estimadas
FROM pg_stat_user_tables
ORDER BY seq_scan DESC NULLS LAST;
"""

if DB_DISPONIVEL:
    with engine.connect() as conn:
        rows = conn.execute(text(SQL_SCAN_RATIO)).fetchall()

    print(f"{'Tabela':<25} {'Seq Scan':>10} {'Idx Scan':>10} {'Idx%':>7} {'Linhas':>10}")
    print("-" * 70)
    for row in rows:
        idx_pct = float(row[3]) if row[3] is not None else 0.0
        alerta = "  << precisa de índice?" if row[4] and row[4] > 1000 and idx_pct < 50 else ""
        print(
            f"{row[0]:<25} {row[1]:>10} {str(row[2] or 0):>10}"
            f" {idx_pct:>6.1f}% {row[4]:>10}{alerta}"
        )

Tabela                      Seq Scan   Idx Scan    Idx%     Linhas
----------------------------------------------------------------------
consultas_externas             51004          0    0.0%          0
parcelas                       51004          0    0.0%          0
decisoes_log                   51004          0    0.0%          0
propostas                      25020     173105   87.4%      10000
clientes                          10      86006  100.0%      10000
bureau_consultas                   6          5   45.5%          6
etapas_fluxo                       5          8   61.5%         10
clientes_historico                 4          0    0.0%          0
analistas                          4          7   63.6%          3
documentos_audit                   3          0    0.0%          3
propostas_audit                    3          2   40.0%          8
documentos                         3          1   25.0%          2
bureau_consultas_2025_01           1          0    0.0%   

In [12]:
# Identificar índices que nunca foram usados (candidatos a remoção)
SQL_UNUSED_IDX = """
SELECT
    schemaname,
    relname AS tablename,
    indexrelname AS indexname,
    idx_scan,
    pg_size_pretty(pg_relation_size(indexrelid)) AS tamanho
FROM pg_stat_user_indexes
WHERE idx_scan = 0
  AND indexrelname NOT LIKE '%_pkey'   -- excluir PKs (não podem ser removidas)
ORDER BY pg_relation_size(indexrelid) DESC;
"""

if DB_DISPONIVEL:
    with engine.connect() as conn:
        rows = conn.execute(text(SQL_UNUSED_IDX)).fetchall()

    if rows:
        print("Índices com zero usos (candidatos a remoção):")
        print(f"{'Schema':<12} {'Tabela':<20} {'Índice':<35} {'Usos':>6} {'Tamanho':>10}")
        print("-" * 90)
        for row in rows:
            print(f"{row[0]:<12} {row[1]:<20} {row[2]:<35} {row[3]:>6} {str(row[4]):>10}")
    else:
        print("Nenhum índice não utilizado encontrado (ou banco reiniciado recentemente).")
        print("Nota: pg_stat_user_indexes é resetado ao reiniciar o PostgreSQL.")
        print("Em produção, verifique após pelo menos 1 semana de uso normal.")

Índices com zero usos (candidatos a remoção):
Schema       Tabela               Índice                                Usos    Tamanho
------------------------------------------------------------------------------------------
public       clientes             clientes_cpf_key                         0    1440 kB
public       bureau_consultas     idx_bureau_response_gin                  0      24 kB
public       analistas            analistas_matricula_key                  0      16 kB
public       analistas            analistas_email_key                      0      16 kB
public       documentos           idx_documentos_proposta                  0      16 kB
public       documentos           idx_documentos_pendentes                 0      16 kB
public       documentos_audit     idx_audit_documento_id                   0      16 kB
public       clientes_historico   clientes_historico_email_idx             0 8192 bytes
public       clientes_historico   clientes_historico_cpf_idx1          

## 6. Caso Real: Sistema que Trava às 10h

### Cenário

O sistema de crédito funciona bem durante a madrugada. Mas às 10h, quando todos os analistas abrem a fila de propostas, o sistema para de responder.

Os sintomas:
- Requests com timeout após 30 segundos
- Aplicação loga: `QueuePool limit of size 10 overflow 5 reached, connection timed out`
- DBA encontra 50+ conexões ativas no banco, todas em `idle in transaction`

### Diagnóstico passo a passo

In [13]:
# Passo 1: Verificar o que está acontecendo agora em pg_stat_activity
SQL_DIAGNOSTICO = """
SELECT
    state,
    wait_event_type,
    wait_event,
    COUNT(*)                                                  AS sessoes,
    MAX(EXTRACT(EPOCH FROM (now() - query_start)))::int       AS max_segundos,
    LEFT(query, 100)                                          AS query_sample
FROM pg_stat_activity
WHERE pid <> pg_backend_pid()
  AND state IS NOT NULL
GROUP BY state, wait_event_type, wait_event, LEFT(query, 100)
ORDER BY sessoes DESC;
"""

# O que esperar em um sistema com problema:
# state='active', wait_event='Lock'   -> sessões bloqueadas aguardando lock
# state='idle in transaction'         -> transação aberta mas não fazendo nada
# state='active', wait_event=None     -> queries rodando normalmente

print("Diagnóstico 1: Estado das conexões ativas")
print("Em um sistema com problema às 10h, você veria algo como:")
print()
print(f"{'State':<25} {'Wait Type':<20} {'Wait Event':<20} {'Sessões':>7} {'MaxSeg':>7}")
print("-" * 85)
# Situação hipotética de problema
cenario_problema = [
    ("active",               "Lock",   "relation",   45,  28, "SELECT id, numero, valor FROM propostas WHERE status='pendente'"),
    ("idle in transaction",  None,     None,          8,  120, "BEGIN"),
    ("active",               "Client", "ClientRead",  3,   2, "COMMIT"),
]
for row in cenario_problema:
    print(f"{row[0]:<25} {str(row[1] or ''):<20} {str(row[2] or ''):<20} {row[3]:>7} {row[4]:>7}")

print()
print("Interpretação:")
print("  - 45 sessões travadas aguardando lock em 'relation'")
print("  - 8 sessões com transações abertas há 2+ minutos (provavelmente a causa)")
print("  - Essas 8 sessões estão segurando locks que as 45 precisam")

Diagnóstico 1: Estado das conexões ativas
Em um sistema com problema às 10h, você veria algo como:

State                     Wait Type            Wait Event           Sessões  MaxSeg
-------------------------------------------------------------------------------------
active                    Lock                 relation                  45      28
idle in transaction                                                       8     120
active                    Client               ClientRead                 3       2

Interpretação:
  - 45 sessões travadas aguardando lock em 'relation'
  - 8 sessões com transações abertas há 2+ minutos (provavelmente a causa)
  - Essas 8 sessões estão segurando locks que as 45 precisam


In [14]:
# Passo 2: Verificar o plano da query mais executada
# A query das 45 sessões bloqueadas: SELECT ... WHERE status='pendente'

if DB_DISPONIVEL:
    # Inserir propostas pendentes para o EXPLAIN ter dados
    with engine.begin() as conn:
        conn.execute(text("UPDATE propostas SET status = 'pendente' WHERE id % 4 = 0"))

    QUERY_PROBLEMA = """
    EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT)
    SELECT id, numero, valor, created_at
    FROM propostas
    WHERE status = 'pendente'
    ORDER BY created_at
    LIMIT 50;
    """

    with engine.connect() as conn:
        rows = conn.execute(text(QUERY_PROBLEMA)).fetchall()

    print("Plano ANTES da correção:")
    for row in rows:
        print(" ", row[0])

    seq_scan_encontrado = any("Seq Scan" in str(row[0]) for row in rows)
    print()
    if seq_scan_encontrado:
        print("PROBLEMA CONFIRMADO: Seq Scan na tabela propostas")
        print("Com 10.000 propostas, cada query varre a tabela inteira")
        print("Com 50 analistas abrindo a fila ao mesmo tempo = 50 Seq Scans simultâneos")

Plano ANTES da correção:
  Limit  (cost=0.25..4.26 rows=1 width=36) (actual time=0.281..0.306 rows=50 loops=1)
    Buffers: shared hit=119
    ->  Index Scan using idx_propostas_pendentes_criacao on propostas  (cost=0.25..4.26 rows=1 width=36) (actual time=0.280..0.301 rows=50 loops=1)
          Buffers: shared hit=119
  Planning:
    Buffers: shared hit=37
  Planning Time: 0.288 ms
  Execution Time: 0.332 ms



In [15]:
# Passo 3: Solução - Índice parcial para propostas pendentes
# Índice parcial: indexa apenas as linhas WHERE status='pendente'
# Muito menor e mais eficiente que um índice em toda a coluna status

# CREATE INDEX CONCURRENTLY não pode rodar dentro de uma transação.
# Usamos autocommit via psycopg2 ou isolation_level="AUTOCOMMIT" no SQLAlchemy.

if DB_DISPONIVEL:
    conn_raw = psycopg2.connect(DSN)
    conn_raw.autocommit = True
    cur = conn_raw.cursor()
    cur.execute("""
        CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_propostas_pendentes_criacao
        ON propostas (created_at)
        WHERE status = 'pendente';
    """)
    cur.close()
    conn_raw.close()
    print("Índice parcial criado: idx_propostas_pendentes_criacao")

    # Verificar o plano após o índice
    with engine.connect() as conn:
        rows = conn.execute(text(QUERY_PROBLEMA)).fetchall()

    print("\nPlano APÓS a correção:")
    for row in rows:
        print(" ", row[0])

    index_scan_encontrado = any("Index" in str(row[0]) for row in rows)
    print()
    if index_scan_encontrado:
        print("CORRIGIDO: Planner usa o índice parcial")
        print("Agora cada query lê apenas as linhas pendentes, não a tabela inteira")

Índice parcial criado: idx_propostas_pendentes_criacao

Plano APÓS a correção:
  Limit  (cost=0.25..4.26 rows=1 width=36) (actual time=0.041..0.061 rows=50 loops=1)
    Buffers: shared hit=8
    ->  Index Scan using idx_propostas_pendentes_criacao on propostas  (cost=0.25..4.26 rows=1 width=36) (actual time=0.040..0.056 rows=50 loops=1)
          Buffers: shared hit=8
  Planning Time: 0.087 ms
  Execution Time: 0.095 ms

CORRIGIDO: Planner usa o índice parcial
Agora cada query lê apenas as linhas pendentes, não a tabela inteira


In [16]:
# Passo 4: Configurar connection pooling adequado
# Problema: 50 analistas * N conexões por instância da app = overflow

print("Configuração ANTES (típico de quem não pensou em produção):")
print("  pool_size    = 1  (default SQLAlchemy sem configuração)")
print("  max_overflow = 10 (default)")
print("  pool_timeout = 30 (30s antes de dar erro)")
print("  Com 50 analistas: 50 conexões diretas ao banco -> PostgreSQL fica sobrecarregado")
print()

# Servidor de produção hipotético: 4 cores, SSD
core_count = 4
spindle_count = 1  # SSD é contado como 1
pool_size_recomendado = (core_count * 2) + spindle_count
max_overflow_recomendado = pool_size_recomendado // 2

print("Configuração CORRIGIDA:")
print(f"  Fórmula: ({core_count} cores * 2) + {spindle_count} spindle = {pool_size_recomendado} conexões")
print(f"  pool_size    = {pool_size_recomendado}")
print(f"  max_overflow = {max_overflow_recomendado}")
print( "  pool_timeout = 10  (falhar rápido é melhor do que 30s de espera)")
print( "  pool_recycle = 1800")
print()
print("Para 3 réplicas da aplicação:")
print(f"  Conexões totais ao banco = 3 * ({pool_size_recomendado} + {max_overflow_recomendado}) = {3 * (pool_size_recomendado + max_overflow_recomendado)}")
print( "  Dentro do limite seguro para PostgreSQL")
print()
print("Alternativa para escala maior: PgBouncer na frente do PostgreSQL")
print("  App abre conexões com PgBouncer (porta 6432)")
print("  PgBouncer mantém pool fixo de 25 conexões reais com PostgreSQL")
print("  Aplicação pode ter 1000+ conexões lógicas sem sobrecarregar o banco")

Configuração ANTES (típico de quem não pensou em produção):
  pool_size    = 1  (default SQLAlchemy sem configuração)
  max_overflow = 10 (default)
  pool_timeout = 30 (30s antes de dar erro)
  Com 50 analistas: 50 conexões diretas ao banco -> PostgreSQL fica sobrecarregado

Configuração CORRIGIDA:
  Fórmula: (4 cores * 2) + 1 spindle = 9 conexões
  pool_size    = 9
  max_overflow = 4
  pool_timeout = 10  (falhar rápido é melhor do que 30s de espera)
  pool_recycle = 1800

Para 3 réplicas da aplicação:
  Conexões totais ao banco = 3 * (9 + 4) = 39
  Dentro do limite seguro para PostgreSQL

Alternativa para escala maior: PgBouncer na frente do PostgreSQL
  App abre conexões com PgBouncer (porta 6432)
  PgBouncer mantém pool fixo de 25 conexões reais com PostgreSQL
  Aplicação pode ter 1000+ conexões lógicas sem sobrecarregar o banco


In [17]:
# Passo 5: Medir o impacto da correção
# Antes: Seq Scan + sem pool adequado
# Depois: Index Scan + pool configurado

if DB_DISPONIVEL:
    QUERY_FILA = """
        SELECT id, numero, valor, created_at
        FROM propostas
        WHERE status = 'pendente'
        ORDER BY created_at
        LIMIT 50;
    """

    # Simular 100 execuções para percentis estatisticamente significativos
    N_EXEC = 100
    tempos = []

    for _ in range(N_EXEC):
        t0 = time.time()
        with engine.connect() as conn:
            rows = conn.execute(text(QUERY_FILA)).fetchall()
        tempos.append((time.time() - t0) * 1000)

    avg = sum(tempos) / len(tempos)
    p95 = sorted(tempos)[int(len(tempos) * 0.95)]
    p99 = sorted(tempos)[int(len(tempos) * 0.99)]

    print(f"Performance da query de fila (com índice parcial, {N_EXEC} execuções):")
    print(f"  Média  : {avg:.2f}ms")
    print(f"  P95    : {p95:.2f}ms")
    print(f"  P99    : {p99:.2f}ms")
    print(f"  Linhas retornadas: {len(rows)}")
    print()
    print("Próximo passo (fora do escopo deste notebook):")
    print("  Cache em Redis para a query de fila (ver Módulo 4)")
    print("  TTL de 30s: analistas verão dados com atraso aceitável")
    print("  Redução de carga: de 50 queries/min para 2 queries/min no banco")

Performance da query de fila (com índice parcial, 100 execuções):
  Média  : 0.48ms
  P95    : 0.85ms
  P99    : 1.45ms
  Linhas retornadas: 50

Próximo passo (fora do escopo deste notebook):
  Cache em Redis para a query de fila (ver Módulo 4)
  TTL de 30s: analistas verão dados com atraso aceitável
  Redução de carga: de 50 queries/min para 2 queries/min no banco


## Referência Rápida

### Checklist de produção

- [ ] Pool configurado: `pool_size = (cores * 2) + spindles`, `pool_pre_ping=True`
- [ ] PgBouncer em `transaction` mode para APIs REST
- [ ] `log_min_duration_statement` configurado (1000ms é um bom default)
- [ ] Monitorar `n_dead_tup` em `pg_stat_user_tables`; alertar se `dead_pct > 20%`

### Quando algo trava

1. `pg_stat_activity` agrupado por `state` + `wait_event` -- revela locks e sessões ociosas em transação
2. `EXPLAIN (ANALYZE, BUFFERS)` na query suspeita -- confirma Seq Scan ou lock contention
3. Índice parcial (`WHERE status = 'pendente'`) resolve queries de fila de trabalho sem indexar a tabela toda

### Carga de dados

`COPY` para volume (50-200x vs INSERT). `execute_values` com `page_size=1000` para batch com lógica Python. `ON CONFLICT` (UPSERT) no banco, não em Python.

### Locks

Advisory locks (`pg_try_advisory_lock`) coordenam workers sem alterar schema. Para múltiplos locks, adquira sempre na mesma ordem. `VACUUM FULL` exige `ACCESS EXCLUSIVE` -- agende em janela de manutenção.

### O que cada view responde

- **`pg_stat_activity`**: quem está conectado, fazendo o quê, há quanto tempo
- **`pg_stat_user_tables`**: a tabela está inchada? Os índices estão sendo usados?
- **`pg_stat_user_indexes`**: esse índice justifica o custo de manutenção?
- **`pg_stat_statements`**: quais queries consomem mais tempo total no banco?

## Exercícios

### Exercício 1: Diagnóstico de dead tuples

Insira 5.000 clientes via `execute_values`, atualize a renda de quem ganha menos de 3000, monitore dead tuples antes e depois de `VACUUM ANALYZE`.

### Exercício 2: UPSERT em escala

Implemente `upsert_propostas(propostas, metodo)` com dois caminhos:
- `execute_values` com `ON CONFLICT DO UPDATE`
- `COPY` para tabela temporária + `INSERT ... ON CONFLICT` da staging para a tabela final

Benchmark com 10.000 propostas onde 50% já existem.

### Exercício 3: Função de diagnóstico de produção

Implemente `diagnostico_completo(engine)` que retorna um score de 0-100 baseado em: sessões `idle in transaction` > 60s, tabelas com `dead_pct > 10%`, índices não usados (exceto PKs), e tabelas com mais de 1.000 linhas onde `seq_scan > idx_scan`.

### Resposta - Exercício 1

In [18]:
# Exercício 1: Diagnóstico de dead tuples
if DB_DISPONIVEL:
    conn_raw = psycopg2.connect(DSN)
    conn_raw.autocommit = True
    cur = conn_raw.cursor()

    # Limpar e reinserir clientes
    cur.execute("DELETE FROM parcelas")
    cur.execute("DELETE FROM decisoes_log")
    cur.execute("DELETE FROM consultas_externas")
    cur.execute("DELETE FROM propostas")
    cur.execute("DELETE FROM clientes")

    # Inserir 5.000 clientes via execute_values
    clientes_data = [
        (f"{20_000 + i:011d}", f"ExCliente {i}", round(random.uniform(1000, 10000), 2))
        for i in range(5000)
    ]

    t0 = time.time()
    psycopg2.extras.execute_values(
        cur,
        "INSERT INTO clientes (cpf, nome, renda_mensal) VALUES %s",
        clientes_data,
        page_size=1000,
    )
    t_insert = time.time() - t0
    print(f"5.000 clientes inseridos via execute_values em {t_insert:.3f}s")

    # Atualizar estatísticas para baseline
    cur.execute("ANALYZE clientes")
    cur.execute("SELECT n_live_tup, n_dead_tup FROM pg_stat_user_tables WHERE relname = 'clientes'")
    antes = cur.fetchone()
    print(f"\nAntes do UPDATE: {antes[0]} vivos, {antes[1]} mortos")

    # Atualizar renda de quem ganha menos de 3000
    cur.execute("UPDATE clientes SET renda_mensal = 3000 WHERE renda_mensal < 3000")
    rows_updated = cur.rowcount
    print(f"Linhas atualizadas (renda < 3000): {rows_updated}")

    # Verificar dead tuples após UPDATE
    cur.execute("ANALYZE clientes")
    cur.execute("SELECT n_live_tup, n_dead_tup FROM pg_stat_user_tables WHERE relname = 'clientes'")
    depois = cur.fetchone()
    print(f"Após UPDATE:     {depois[0]} vivos, {depois[1]} mortos")

    if depois[0] and depois[1]:
        dead_pct = 100.0 * depois[1] / (depois[0] + depois[1])
        print(f"Dead tuple %:    {dead_pct:.1f}%")

    # VACUUM ANALYZE para recuperar espaço
    cur.execute("VACUUM ANALYZE clientes")
    cur.execute("SELECT n_live_tup, n_dead_tup FROM pg_stat_user_tables WHERE relname = 'clientes'")
    apos_vacuum = cur.fetchone()
    print(f"Após VACUUM:     {apos_vacuum[0]} vivos, {apos_vacuum[1]} mortos")

    cur.close()
    conn_raw.close()

5.000 clientes inseridos via execute_values em 0.062s

Antes do UPDATE: 5000 vivos, 10000 mortos
Linhas atualizadas (renda < 3000): 1068
Após UPDATE:     5000 vivos, 10000 mortos
Dead tuple %:    66.7%
Após VACUUM:     5000 vivos, 0 mortos


### Resposta - Exercício 2

In [19]:
# Exercício 2: UPSERT em escala - execute_values vs COPY + staging table

def upsert_propostas(propostas, metodo, dsn):
    """
    Upsert propostas usando execute_values ou COPY + staging table.
    Retorna tempo em segundos.
    """
    conn = psycopg2.connect(dsn)
    conn.autocommit = False
    cur = conn.cursor()

    t0 = time.time()

    if metodo == "execute_values":
        # execute_values com ON CONFLICT DO UPDATE
        psycopg2.extras.execute_values(
            cur,
            """
            INSERT INTO propostas (numero, cliente_id, valor, status, score)
            VALUES %s
            ON CONFLICT (numero)
            DO UPDATE SET
                status = EXCLUDED.status,
                score  = EXCLUDED.score
            """,
            propostas,
            page_size=1000,
        )
        conn.commit()

    elif metodo == "copy":
        # COPY para tabela temporária + INSERT ... ON CONFLICT da staging
        cur.execute("""
            CREATE TEMP TABLE propostas_staging (
                numero VARCHAR(20),
                cliente_id INT,
                valor NUMERIC(12,2),
                status VARCHAR(20),
                score SMALLINT
            ) ON COMMIT DROP
        """)

        # COPY dados para staging
        buffer = io.StringIO()
        writer = csv.writer(buffer)
        writer.writerows(propostas)
        buffer.seek(0)

        cur.copy_expert(
            "COPY propostas_staging (numero, cliente_id, valor, status, score) FROM STDIN WITH CSV",
            buffer,
        )

        # INSERT da staging para a tabela real com ON CONFLICT
        cur.execute("""
            INSERT INTO propostas (numero, cliente_id, valor, status, score)
            SELECT numero, cliente_id, valor, status, score
            FROM propostas_staging
            ON CONFLICT (numero)
            DO UPDATE SET
                status = EXCLUDED.status,
                score  = EXCLUDED.score
        """)
        conn.commit()

    else:
        raise ValueError(f"Método desconhecido: {metodo}")

    elapsed = time.time() - t0
    cur.close()
    conn.close()
    return elapsed


if DB_DISPONIVEL:
    # Preparar: garantir que existem clientes e propostas base
    conn_raw = psycopg2.connect(DSN)
    conn_raw.autocommit = True
    cur = conn_raw.cursor()

    # Limpar e reinserir dados base
    cur.execute("DELETE FROM parcelas")
    cur.execute("DELETE FROM decisoes_log")
    cur.execute("DELETE FROM consultas_externas")
    cur.execute("DELETE FROM propostas")
    cur.execute("DELETE FROM clientes")
    cur.execute("""
        INSERT INTO clientes (cpf, nome, renda_mensal)
        SELECT LPAD(generate_series::text, 11, '0'), 'Cliente ' || generate_series,
               (random() * 10000 + 1000)::numeric(12,2)
        FROM generate_series(1, 10000)
    """)
    cur.execute("SELECT MIN(id), MAX(id) FROM clientes")
    min_id, max_id = cur.fetchone()
    cur.close()
    conn_raw.close()

    # Gerar 10.000 propostas; inserir metade para que 50% já existam
    propostas_existentes = [
        (f"UPSERT-{i:08d}", random.randint(min_id, max_id),
         round(random.uniform(500, 50000), 2), "pendente", random.randint(300, 850))
        for i in range(5000)
    ]

    # Inserir primeira metade
    conn_raw = psycopg2.connect(DSN)
    cur = conn_raw.cursor()
    psycopg2.extras.execute_values(
        cur,
        "INSERT INTO propostas (numero, cliente_id, valor, status, score) VALUES %s",
        propostas_existentes,
        page_size=1000,
    )
    conn_raw.commit()
    cur.close()
    conn_raw.close()

    # Gerar as 10.000 propostas para upsert (5000 existentes + 5000 novas)
    propostas_novas = [
        (f"UPSERT-{5000 + i:08d}", random.randint(min_id, max_id),
         round(random.uniform(500, 50000), 2), "aprovada", random.randint(300, 850))
        for i in range(5000)
    ]
    # Alterar status das existentes para simular update
    propostas_update = [
        (p[0], p[1], p[2], "aprovada", 999)
        for p in propostas_existentes
    ]
    todas = propostas_update + propostas_novas

    # Benchmark execute_values
    t_ev = upsert_propostas(todas, "execute_values", DSN)
    print(f"execute_values UPSERT (10.000 linhas, 50% conflito): {t_ev:.3f}s  ({10000/t_ev:.0f} linhas/s)")

    # Resetar para benchmark do COPY (recriar estado original)
    conn_raw = psycopg2.connect(DSN)
    cur = conn_raw.cursor()
    cur.execute("DELETE FROM propostas")
    psycopg2.extras.execute_values(
        cur,
        "INSERT INTO propostas (numero, cliente_id, valor, status, score) VALUES %s",
        propostas_existentes,
        page_size=1000,
    )
    conn_raw.commit()
    cur.close()
    conn_raw.close()

    # Benchmark COPY + staging
    t_copy = upsert_propostas(todas, "copy", DSN)
    print(f"COPY + staging   UPSERT (10.000 linhas, 50% conflito): {t_copy:.3f}s  ({10000/t_copy:.0f} linhas/s)")

    print(f"\nCOPY + staging é {t_ev/t_copy:.1f}x mais rápido que execute_values para UPSERT")

execute_values UPSERT (10.000 linhas, 50% conflito): 0.275s  (36390 linhas/s)
COPY + staging   UPSERT (10.000 linhas, 50% conflito): 0.191s  (52428 linhas/s)

COPY + staging é 1.4x mais rápido que execute_values para UPSERT


### Resposta - Exercício 3

In [20]:
# Exercício 3: Função de diagnóstico de produção

def diagnostico_completo(engine):
    """
    Executa verificações de saúde no banco e retorna um score de 0-100.
    Penalidades:
      - Sessões idle in transaction > 60s: -10 por sessão
      - Tabelas com dead_pct > 10%: -10 por tabela
      - Índices não usados (exceto PKs): -5 por índice
      - Tabelas >1000 linhas com seq_scan > idx_scan: -10 por tabela
    """
    score = 100
    problemas = []

    with engine.connect() as conn:
        # 1. Conexões idle in transaction > 60s
        rows = conn.execute(text("""
            SELECT pid, usename,
                   EXTRACT(EPOCH FROM (now() - query_start))::int AS segundos,
                   LEFT(query, 80) AS query
            FROM pg_stat_activity
            WHERE state = 'idle in transaction'
              AND EXTRACT(EPOCH FROM (now() - query_start)) > 60
              AND pid <> pg_backend_pid()
        """)).fetchall()

        total_sessoes = conn.execute(text(
            "SELECT COUNT(*) FROM pg_stat_activity WHERE pid <> pg_backend_pid()"
        )).scalar()

        print(f"=== CONEXÕES ===")
        print(f"Total de sessões ativas: {total_sessoes}")
        if rows:
            for r in rows:
                print(f"  ALERTA: PID {r[0]} ({r[1]}) idle in transaction há {r[2]}s")
                problemas.append(f"Sessão {r[0]} idle in transaction há {r[2]}s")
            score -= 10 * len(rows)
        else:
            print("  Nenhuma sessão idle in transaction > 60s")

        # 2. Tabelas com bloat (dead_pct > 10%)
        rows = conn.execute(text("""
            SELECT relname, n_live_tup, n_dead_tup,
                   ROUND(100.0 * n_dead_tup / NULLIF(n_live_tup + n_dead_tup, 0), 1) AS dead_pct
            FROM pg_stat_user_tables
            WHERE n_dead_tup > 0
              AND 100.0 * n_dead_tup / NULLIF(n_live_tup + n_dead_tup, 0) > 10
            ORDER BY n_dead_tup DESC
        """)).fetchall()

        print(f"\n=== BLOAT (dead_pct > 10%) ===")
        if rows:
            for r in rows:
                print(f"  ALERTA: {r[0]} - {r[2]} dead tuples ({float(r[3]):.1f}%)")
                problemas.append(f"Tabela {r[0]} com {float(r[3]):.1f}% dead tuples")
            score -= 10 * len(rows)
        else:
            print("  Nenhuma tabela com bloat significativo")

        # 3. Índices não usados (exceto PKs)
        # pg_stat_user_indexes usa relname e indexrelname, não tablename/indexname
        rows = conn.execute(text("""
            SELECT relname, indexrelname, idx_scan,
                   pg_size_pretty(pg_relation_size(indexrelid)) AS tamanho
            FROM pg_stat_user_indexes
            WHERE idx_scan = 0
              AND indexrelname NOT LIKE '%_pkey'
            ORDER BY pg_relation_size(indexrelid) DESC
        """)).fetchall()

        print(f"\n=== ÍNDICES NÃO USADOS ===")
        if rows:
            for r in rows:
                print(f"  {r[0]}.{r[1]} ({r[3]}) - 0 usos")
                problemas.append(f"Índice não usado: {r[1]}")
            score -= 5 * len(rows)
        else:
            print("  Todos os índices estão sendo usados")

        # 4. Tabelas sem índice eficiente (>1000 linhas, seq_scan > idx_scan)
        rows = conn.execute(text("""
            SELECT relname, seq_scan, idx_scan, n_live_tup
            FROM pg_stat_user_tables
            WHERE n_live_tup > 1000
              AND seq_scan > COALESCE(idx_scan, 0)
            ORDER BY seq_scan DESC
        """)).fetchall()

        print(f"\n=== TABELAS SEM ÍNDICE EFICIENTE ===")
        if rows:
            for r in rows:
                print(f"  {r[0]}: {r[1]} seq scans vs {r[2] or 0} idx scans ({r[3]} linhas)")
                problemas.append(f"Tabela {r[0]} com mais seq scans que idx scans")
            score -= 10 * len(rows)
        else:
            print("  Todas as tabelas grandes estão usando índices")

    # Score final (mínimo 0)
    score = max(0, score)
    status = "SAUDÁVEL" if score >= 80 else "ATENÇÃO" if score >= 50 else "CRÍTICO"

    print(f"\n{'='*40}")
    print(f"SCORE: {score}/100 ({status})")
    if problemas:
        print(f"Problemas encontrados: {len(problemas)}")
        for p in problemas:
            print(f"  - {p}")
    print(f"{'='*40}")

    return score


# Executar diagnóstico
if DB_DISPONIVEL:
    score = diagnostico_completo(engine)
    print(f"\nScore retornado: {score}")

=== CONEXÕES ===
Total de sessões ativas: 6
  Nenhuma sessão idle in transaction > 60s

=== BLOAT (dead_pct > 10%) ===
  ALERTA: propostas - 22569 dead tuples (69.3%)
  ALERTA: clientes - 6068 dead tuples (28.8%)
  ALERTA: bureau_consultas - 1 dead tuples (14.3%)
  ALERTA: documentos - 1 dead tuples (33.3%)

=== ÍNDICES NÃO USADOS ===
  clientes.clientes_cpf_key (1440 kB) - 0 usos
  bureau_consultas.idx_bureau_response_gin (24 kB) - 0 usos
  analistas.analistas_matricula_key (16 kB) - 0 usos
  analistas.analistas_email_key (16 kB) - 0 usos
  documentos.idx_documentos_proposta (16 kB) - 0 usos
  documentos.idx_documentos_pendentes (16 kB) - 0 usos
  documentos_audit.idx_audit_documento_id (16 kB) - 0 usos
  clientes_historico.clientes_historico_email_idx (8192 bytes) - 0 usos
  clientes_historico.clientes_historico_cpf_idx1 (8192 bytes) - 0 usos
  clientes_historico.clientes_historico_cpf_idx (8192 bytes) - 0 usos

=== TABELAS SEM ÍNDICE EFICIENTE ===
  Todas as tabelas grandes estão us